In [2]:
import pandas as pd
import json
import os
from tqdm import tqdm

def parse_annotation_line(line):
    """
    Парсит строку аннотации в формате: путь_к_изображению\t[json_аннотации]
    
    Args:
        line (str): Строка из файла аннотаций
        
    Returns:
        dict: Словарь с распарсенными данными:
            {
                'image_path': str,  # путь к изображению
                'annotations': list  # список аннотаций (словарей)
            }
            
    Пример возвращаемого значения:
    {
        'image_path': 'rgb/img1483.jpg',
        'annotations': [
            {
                'transcription': 'COUNTY',
                'points': [[328,482], [349,482], ...]
            },
            ...
        ]
    }
    """
    try:
        # Разделяем строку по табуляции
        parts = line.strip().split('\t')
        if len(parts) != 2:
            raise ValueError("Некорректный формат строки - отсутствует табуляция")
            
        image_path, annotations_json = parts
        
        # Парсим JSON
        annotations = json.loads(annotations_json)
        
        # Проверяем структуру аннотаций
        if not isinstance(annotations, list):
            raise ValueError("Аннотации должны быть списком")
            
        for ann in annotations:
            if 'transcription' not in ann or 'points' not in ann:
                raise ValueError("Каждая аннотация должна содержать 'transcription' и 'points'")
                
        return {
            'image_path': image_path,
            'annotations': annotations
        }
        
    except json.JSONDecodeError as e:
        raise ValueError(f"Ошибка парсинга JSON: {str(e)}")
    except Exception as e:
        raise ValueError(f"Ошибка обработки строки: {str(e)}")


# Пример использования
#line = 'rgb/img1483.jpg\t[{"transcription": "COUNTY", "points": [[328,482], [349,482], [370,483]]}]'
#parsed = parse_annotation_line(line)
#print(parsed)

train_annotations_path = 'Dataset2/total_text/train/train.txt'
train_images_dir = 'Dataset2/total_text/train/rgb'

# Собираем все данные
data = []

with open(train_annotations_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Processing annotations"):
        parsed = parse_annotation_line(line)
        if not parsed:
            continue
            
        image_path = parsed['image_path']
        img_id = os.path.splitext(os.path.basename(image_path))[0]
        full_image_path = os.path.join(train_images_dir, os.path.basename(image_path))
        
        for ann in parsed['annotations']:
            data.append({
                'img_id': img_id,
                'label': ann['transcription'],
                'bbox': ann['points'],
                'Image_path': full_image_path
            })

# Создаем DataFrame
train_df = pd.DataFrame(data, columns=['img_id', 'label', 'bbox', 'Image_path'])

# Сохраняем для проверки
train_df.to_csv('d2_train.csv', index=False)
print(f"Парсинг завершен. Получено {len(train_df)} записей.")
print(train_df.head())

Processing annotations: 1255it [00:00, 10080.57it/s]


Парсинг завершен. Получено 11144 записей.
    img_id      label                                               bbox  \
0  img1483     COUNTY  [[328.0, 482.0], [349.0, 482.0], [370.0, 483.0...   
1  img1483        TOP  [[224.0, 275.0], [237.0, 277.0], [250.0, 278.0...   
2  img1483    WELCOME  [[59.0, 79.0], [93.0, 79.0], [128.0, 80.0], [1...   
3  img1483   HISTORIC  [[390.0, 102.0], [419.0, 103.0], [449.0, 104.0...   
4  img1483  LEADVILLE  [[51.0, 166.0], [144.0, 170.0], [237.0, 173.0]...   

                                  Image_path  
0  Dataset2/total_text/train/rgb/img1483.jpg  
1  Dataset2/total_text/train/rgb/img1483.jpg  
2  Dataset2/total_text/train/rgb/img1483.jpg  
3  Dataset2/total_text/train/rgb/img1483.jpg  
4  Dataset2/total_text/train/rgb/img1483.jpg  


In [3]:
train_df

,img_id,label,bbox,Image_path
0,img1483,COUNTY,"[[328.0, 482.0], [349.0, 482.0], [370.0, 483.0...",Dataset2/total_text/train/rgb/img1483.jpg
1,img1483,TOP,"[[224.0, 275.0], [237.0, 277.0], [250.0, 278.0...",Dataset2/total_text/train/rgb/img1483.jpg
2,img1483,WELCOME,"[[59.0, 79.0], [93.0, 79.0], [128.0, 80.0], [1...",Dataset2/total_text/train/rgb/img1483.jpg
3,img1483,HISTORIC,"[[390.0, 102.0], [419.0, 103.0], [449.0, 104.0...",Dataset2/total_text/train/rgb/img1483.jpg
4,img1483,LEADVILLE,"[[51.0, 166.0], [144.0, 170.0], [237.0, 173.0]...",Dataset2/total_text/train/rgb/img1483.jpg
...,...,...,...,...
11139,img1245,your,"[[494.0, 624.0], [496.0, 621.0], [498.0, 619.0...",Dataset2/total_text/train/rgb/img1245.jpg
11140,img1245,UPoints,"[[509.0, 608.0], [512.0, 605.0], [516.0, 602.0...",Dataset2/total_text/train/rgb/img1245.jpg
11141,img1245,balance,"[[531.0, 589.0], [536.0, 586.0], [540.0, 583.0...",Dataset2/total_text/train/rgb/img1245.jpg
11142,img1245,###,"[[558.0, 569.0], [558.0, 568.0], [559.0, 568.0...",Dataset2/total_text/train/rgb/img1245.jpg


In [4]:
# кол во картинок 
# количество боксов на картинке
# 

import pandas as pd
import json
import os
from tqdm import tqdm

# Пути к тестовым данным
test_annotations_path = 'Dataset2/total_text/test/test.txt'
test_images_dir = 'Dataset2/total_text/test/rgb'

# Собираем тестовые данные
test_data = []

with open(test_annotations_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Processing test annotations"):
        parsed = parse_annotation_line(line)
        if not parsed:
            continue
            
        image_path = parsed['image_path']
        img_id = os.path.splitext(os.path.basename(image_path))[0]
        full_image_path = os.path.join(test_images_dir, os.path.basename(image_path))
        
        for ann in parsed['annotations']:
            test_data.append({
                'img_id': img_id,
                'label': ann['transcription'],
                'bbox': ann['points'],
                'Image_path': full_image_path
            })

# Создаем DataFrame для тестовых данных
df_test = pd.DataFrame(test_data, columns=['img_id', 'label', 'bbox', 'Image_path'])

# Сохраняем результат
df_test.to_csv('d2_test.csv', index=False)
print(f"Тестовые данные обработаны. Получено {len(df_test)} записей.")
print(df_test.head())

Processing test annotations: 300it [00:00, 3455.26it/s]

Тестовые данные обработаны. Получено 2543 записей.
   img_id   label                                               bbox  \
0  img589  MARKET  [[53.0, 182.0], [55.3, 166.6], [57.5, 151.1], ...   
1  img589   FRESH  [[152.0, 53.0], [169.7, 48.6], [187.5, 45.2], ...   
2  img589   FOODS  [[286.0, 77.0], [297.7, 89.4], [309.3, 101.7],...   
3  img995     obe  [[332.0, 430.0], [349.8, 430.0], [367.6, 430.0...   
4  img995      Fo  [[93.0, 433.0], [105.5, 433.0], [118.0, 433.0]...   

                                Image_path  
0  Dataset2/total_text/test/rgb/img589.jpg  
1  Dataset2/total_text/test/rgb/img589.jpg  
2  Dataset2/total_text/test/rgb/img589.jpg  
3  Dataset2/total_text/test/rgb/img995.jpg  
4  Dataset2/total_text/test/rgb/img995.jpg  


In [19]:
df_test.head(25)

# инт - абсолютные  
# флоат - относительный 

,img_id,label,bbox,Image_path
0,img589,MARKET,"[[53.0, 182.0], [55.3, 166.6], [57.5, 151.1], ...",Dataset2/total_text/test/rgb/img589.jpg
1,img589,FRESH,"[[152.0, 53.0], [169.7, 48.6], [187.5, 45.2], ...",Dataset2/total_text/test/rgb/img589.jpg
2,img589,FOODS,"[[286.0, 77.0], [297.7, 89.4], [309.3, 101.7],...",Dataset2/total_text/test/rgb/img589.jpg
3,img995,obe,"[[332.0, 430.0], [349.8, 430.0], [367.6, 430.0...",Dataset2/total_text/test/rgb/img995.jpg
4,img995,Fo,"[[93.0, 433.0], [105.5, 433.0], [118.0, 433.0]...",Dataset2/total_text/test/rgb/img995.jpg
5,img995,chinese_text,"[[546.0, 225.0], [549.1, 225.0], [552.2, 225.0...",Dataset2/total_text/test/rgb/img995.jpg
6,img995,GOLDEN,"[[159.0, 398.0], [165.5, 403.4], [171.9, 408.8...",Dataset2/total_text/test/rgb/img995.jpg
7,img995,GATE,"[[218.0, 425.0], [224.1, 426.8], [230.3, 428.7...",Dataset2/total_text/test/rgb/img995.jpg
8,img995,BRIDGE,"[[266.0, 432.0], [275.8, 431.7], [285.6, 431.3...",Dataset2/total_text/test/rgb/img995.jpg
9,img995,GOLDEN,"[[472.0, 289.0], [485.7, 289.3], [499.4, 289.6...",Dataset2/total_text/test/rgb/img995.jpg
